In [ ]:
from pathlib import Path
import datetime
import re
import torch
import json

In [2]:
from langchain_community.vectorstores import FAISS
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.document_loaders import PyPDFLoader
from langchain_community.document_loaders import TextLoader
from langchain_text_splitters import CharacterTextSplitter
from langchain_classic.output_parsers import StructuredOutputParser, ResponseSchema
from langchain_core.prompts import PromptTemplate
from transformers import AutoTokenizer, AutoModelForCausalLM

C:\Users\Master73\AppData\Local\Temp\ipykernel_6620\2361760475.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.vectorstores import FAISS
d:\Projects\DreamCatcher\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
model_name = "mistralai/Mistral-7B-Instruct-v0.2"

try:
    tokenizer = AutoTokenizer.from_pretrained(model_name)
except Exception as e:
    tokenizer = None
    print(f"Tokenizer load failed: {e}")

device = "cuda" if torch.cuda.is_available() else "cpu"
torch_dtype = torch.float16 if device == "cuda" else torch.float32

try:
    model = AutoModelForCausalLM.from_pretrained(
        model_name,
        torch_dtype=torch_dtype,
    ).to(device)
    model.eval()
except Exception as e:
    model = None
    print(f"Model load failed: {e}")


def generate_text(prompt, max_length=1000, num_return_sequences=1):
    if tokenizer is None or model is None:
        return "Model could not be loaded."
    inputs = tokenizer(prompt, return_tensors="pt").to(device)
    outputs = model.generate(
        **inputs,
        max_length=max_length,
        num_return_sequences=num_return_sequences,
        do_sample=True,
        top_k=50,
        top_p=0.95,
        temperature=0.7,
    )
    return [tokenizer.decode(output, skip_special_tokens=True) for output in outputs][0]

`torch_dtype` is deprecated! Use `dtype` instead!
Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]

In [ ]:
dream_title = ResponseSchema(
    name="dream-title",
    description="A concise meaningful title the discribe the dream. Mainly consists of three or less words."
)
dream_date = ResponseSchema(
    name="dream_date",
    description="The date (in DD-MM-YYYY format) in which the dream happened. {date}"
)
dream_desc = ResponseSchema(
    name="dream_description",
    description="A exact replica of the user inputed dream word for word."
)
dream_symbols = ResponseSchema(
    name="dream_symbols",
    description="A list of all the possible symbolism cotained in the dream."
)
dream_vibes = ResponseSchema(
    name="dream_vibes",
    description="A short list of words that discribe the main feeling (vibe) of the dream."
)

response_schemas = [dream_title,
                    dream_date,
                    dream_desc,
                    dream_symbols, 
                    dream_vibes]

output_parser = StructuredOutputParser.from_response_schemas(response_schemas)
format_instructions = output_parser.get_format_instructions()

In [ ]:
dream_journal_template_prompt = """
You are an expert dream journaler and analyst that extracts dream details based on the user's input.



Extract all qualifications as follows:

dream title
dream date {date}
dream description, including details about what happened, how it felt, and any related ideas
dream symbols
dream vibes, describing the main feeling of the dream


Respond ONLY in Markdown format as follows:
{format_instructions}

Example Input:
"
I had a dream were I was runing away from a mirror
"

Expected output (im markdown):
"
# Dream Title
The Mirror

**Dream Date:** 27-07-2026

## Dream

## Dream Description
I had a dream were I was runing away from a mirror

## Dream Symbols
- Mirror
- Avoidance

## Dream Vibes
- Fear
- Avoidance
"

Now extract from the following input:
"{user_input}"
"""

In [ ]:
def ask_question(query):
    prompt = f"""You are a helpful assistant. Use the following context to answer the question. Question: {query}"""
    
    result = generate_text(prompt)
    return result.strip()

In [ ]:
def extract_json_block(text):
    pattern = r'```json\s*(.*?)\s*```'
    matches = re.findall(pattern, text, re.DOTALL)

    return f"```json\n{matches[-1]}\n```"

In [ ]:
date = datetime.date.today().strftime("%d-%m-%Y")

In [ ]:
user_input = "I had a dream where I was eating then suddenly I turned into a 100m tall gaint!"

prompt = PromptTemplate(
    template=dream_journal_template_prompt,
    input_variables=["user_input", "format_instructions", date]
).format(user_input=user_input, format_instructions=format_instructions)

In [ ]:
answer = ask_question(prompt)
print("\n Answer:", answer.split("Answer:")[-1], "\n")

In [ ]:
json_text = extract_json_block(answer)
print(json_text)

In [ ]:
output_data = output_parser.parse(json_text)
print(output_data)

In [ ]:


# Ensure database directory exists
db_dir = Path('database')
db_dir.mkdir(parents=True, exist_ok=True)

# Try to pretty-format JSON; fall back to raw string if parsing fails",
try:
    obj = json.loads(json_text)
    json_content = json.dumps(obj, indent=2, ensure_ascii=False)
except Exception:
    json_content = json_text

# Write JSON file named after `date`
json_path = db_dir / f\"{date}.json\"
with open(json_path, 'w', encoding='utf-8') as f:
    f.write(json_content)

# Write Markdown file (include JSON in a code fence)
md_path = db_dir / f\"{date}.md\"
with open(md_path, 'w', encoding='utf-8') as f:
    f.write('```json\\n')
    f.write(json_content)
    f.write('\\n```')

print(f\"Saved: {json_path} and {md_path}\")